In [6]:
import numpy as np
import networkx as nx
from scipy.ndimage import binary_dilation
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
from skimage import measure
import skimage as ski
import dask.array as da
from scipy.spatial import ConvexHull, Delaunay, delaunay_plot_2d, convex_hull_plot_2d
import scipy.ndimage as ndi
import ast
from extract_features import subimage
from collections import defaultdict
from misc_utils import fixup_scipy_ndimage_result as fix
from misc_utils import strel_disk
import plotly.express as px
from dask.distributed import Client
import dask.delayed
import seaborn as sns
import scipy.signal
from brieflow_segment_utils import dask_image_log_scale, reconcile_nuclei_cells
from find_neighbors import objects_bounds, neighbors_for_object

In [ ]:
#@dask.delayed
def neighbors_for_object(object_number, min_i, max_i, min_j, max_j, labels_path, strel, distance):
    img = tifffile.memmap(labels_path)
    img = img[::2,::2]
    # img,_,_ = ski.segmentation.relabel_sequential(img, offset=1)
    patch = img[
        min_i : max_i,
        min_j : max_j,
    ]

    #
    # Find the neighbors
    #
    
    patch_mask = patch == (object_number)

    if distance <= 5:
        extended = ndi.binary_dilation(patch_mask, strel)
    else:
        extended = (
            scipy.signal.fftconvolve(patch_mask, strel, mode="same") > 0.5
        )

    neighbors = np.unique(patch[extended])

    neighbors = neighbors[(neighbors != 0) & (neighbors != object_number)]

    if hasattr(neighbors, "compute"):
        neighbors = neighbors.compute()

    return [(int(n), int(object_number)) for n in neighbors]


In [7]:
labels_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif"
nuclei_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_filtered_nuclei.tif"
img_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day24_s12.ome.tif"
foci_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_filtered_foci.tif"
df = pd.read_csv("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/o8p_day24_s12_results_old.csv", index_col="label",converters={"cells_bounds":ast.literal_eval}).drop(columns=["nucleus_percent_touching_1", "nucleus_bounds"])
df["log_foci_count"] = np.log1p(df["foci_count"])
# y0 = 15000:20000
# x0 = 23000:27000
# slice = [22000:29000,15000:27000]
labels_full = tifffile.memmap(labels_path)
labels = labels_full[::2,::2]
img = da.from_zarr(tifffile.imread(img_path, aszarr=True))
foci = da.from_zarr(tifffile.imread(foci_path, aszarr=True))
nuclei = da.from_zarr(tifffile.imread(nuclei_path, aszarr=True))
# labels,_,_ = ski.segmentation.relabel_sequential(labels, offset=1)


In [ ]:
client.close()

In [8]:
nobjects = np.max(labels)
object_indexes = np.arange(nobjects, dtype=np.int32) + 1
distance = 57
strel = strel_disk(distance)

objs = ndi.find_objects(labels, max_label=nobjects)
# objs is a list of slice-tuples, one per label 1..nobjects
# missing labels give None

minimums_i = np.empty(nobjects, dtype=np.int64)
maximums_i = np.empty(nobjects, dtype=np.int64)
minimums_j = np.empty(nobjects, dtype=np.int64)
maximums_j = np.empty(nobjects, dtype=np.int64)

for k, slc in enumerate(objs):
    if slc is None:
        minimums_i[k] = 0
        maximums_i[k] = 0
        minimums_j[k] = 0
        maximums_j[k] = 0
        continue

    si, sj = slc
    minimums_i[k] = max(si.start - distance, 0)
    maximums_i[k] = min(si.stop  + distance, labels.shape[0])
    minimums_j[k] = max(sj.start - distance, 0)
    maximums_j[k] = min(sj.stop  + distance, labels.shape[1])

In [9]:
nobjects = np.max(labels)
object_indexes = np.arange(nobjects, dtype=np.int32) + 1
distance = 57
strel = strel_disk(distance)

minimums_i, maximums_i, minimums_j, maximums_j = objects_bounds(labels, distance)

client = Client()
futures = client.map(
    neighbors_for_object,
    object_indexes.tolist(),
    minimums_i.tolist(),
    maximums_i.tolist(),
    minimums_j.tolist(),
    maximums_j.tolist(),
    [labels_path] * len(object_indexes),
    [strel] * len(object_indexes),
    [distance] * len(object_indexes),
)

result = client.gather(futures)

client.close()

edges = [edge for sublist in result for edge in sublist]

In [10]:
result

[[],
 [(4, 2)],
 [(4, 3)],
 [(2, 4), (3, 4)],
 [(7, 5), (11, 5)],
 [],
 [(5, 7), (8, 7), (11, 7), (12, 7)],
 [(7, 8), (11, 8), (12, 8)],
 [(10, 9)],
 [(9, 10)],
 [(5, 11), (7, 11), (8, 11), (12, 11)],
 [(7, 12), (8, 12), (11, 12)],
 [(14, 13)],
 [(13, 14)],
 [],
 [],
 [(19, 17)],
 [],
 [(17, 19), (20, 19), (21, 19), (26, 19)],
 [(19, 20), (21, 20), (26, 20)],
 [(19, 21), (20, 21), (26, 21)],
 [],
 [],
 [(27, 24), (29, 24), (30, 24)],
 [(28, 25)],
 [(19, 26), (20, 26), (21, 26)],
 [(24, 27), (29, 27), (30, 27)],
 [(25, 28)],
 [(24, 29), (27, 29), (30, 29), (34, 29)],
 [(24, 30), (27, 30), (29, 30), (34, 30)],
 [(36, 31)],
 [],
 [],
 [(29, 34), (30, 34)],
 [],
 [(31, 36)],
 [],
 [],
 [(44, 39)],
 [(41, 40)],
 [(40, 41), (51, 41)],
 [],
 [(50, 43), (53, 43)],
 [(39, 44)],
 [],
 [(49, 46), (55, 46), (61, 46)],
 [(49, 47)],
 [(52, 48), (56, 48)],
 [(46, 49), (47, 49), (55, 49), (61, 49)],
 [(43, 50), (53, 50), (58, 50), (60, 50)],
 [(41, 51)],
 [(48, 52), (56, 52), (57, 52), (64, 52)],
 [(4

In [ ]:
G = nx.Graph()
G.add_nodes_from(object_indexes)
G.add_edges_from(edges)
components = list(nx.connected_components(G))
# nx.write_gml(G, "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12.gml", stringizer=str)
colony_map = {}
i = 1
for comp in components:
    if len(comp) < 9:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = i
        i += 1
df["colony_id"] = df.index.map(colony_map)
np.unique(df["colony_id"].values)

In [ ]:
df

In [ ]:
nuclei, cells = reconcile_nuclei_cells(nuclei, labels_full, how='contained_in_cells')

In [ ]:
points = df[df["colony_id"]==1][['cells_i', 'cells_j']].to_numpy()
hull = ConvexHull(points)
bb = points[hull.vertices]
max = list(map(int, np.max(bb, axis=0)))
min = list(map(int, np.min(bb, axis=0)))
inde = list(map(int, np.append(np.min(bb, axis=0),np.max(bb, axis=0))))
# # draw edges
# for simplex in hull.simplices:
#     ax.plot(points[simplex, 0], points[simplex, 1],
#         color='red', linewidth=1)

In [ ]:
inde = [5000,18000,8000,20000]

In [ ]:
client.close()

In [ ]:
clust1 = subimage(img,inde)
fociclust1 = subimage(foci, inde)
nucleiclust1 = subimage(nuclei, inde)
cytoclust1 = subimage(labels_full, inde)
logclust1 = dask_image_log_scale(clust1)
# clust1 = clust1.compute()
fig, ax = plt.subplots(figsize=(20,20))
ax.set_xticks([])
ax.set_yticks([])
ax.imshow((ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(clust1[0], in_range=(0,3000)),fociclust1),nucleiclust1,mode="thick",color=(1,0,1))))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

ax.scatter(
    df["nucleus_i"],
    df["nucleus_j"],
    c=(df["log_foci_count"]),
    s=5,
    cmap="viridis_r"
)

for colony_id, subdf in df.groupby("colony_id"):
    if colony_id == -1:
        continue
    points = subdf[['cells_i', 'cells_j']].to_numpy()
    hull = ConvexHull(points)

    # draw edges
    for simplex in hull.simplices:
        ax.plot(points[simplex, 0], points[simplex, 1],
                color='red', linewidth=1)
    # compute centroid of hull
    centroid = points[hull.vertices].mean(axis=0)
    # label
    ax.text(
        centroid[0],
        centroid[1],
        str(colony_id),
        color='black',
        fontsize=10,
        ha='center',
        va='center'
    )
    

ax.set_xticks([])
ax.set_yticks([])
ax.invert_xaxis()
plt.show()

In [ ]:
df[["foci_count","cells_area","colony_id"]].groupby("colony_id").describe()

In [ ]:

max_label = labels.max()

label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[labels]

ids = np.unique(colony_img)
ids = ids[ids>0]
cmap = plt.get_cmap("tab20", len(ids))

label_to_color = {
    lab: cmap(i)
    for i, lab in enumerate(ids)
}

# set background / non-colony
label_to_color[-1] = (0.7, 0.7, 0.7, 1)
label_to_color[0] = (0,0,0, 1)

colors_array = np.zeros((max_label + 1, 4))

for lab, color in label_to_color.items():
    colors_array[lab] = color

rgb = colors_array[colony_img]

fig, ax = plt.subplots(figsize=(10,10))
ax.set_xticks([])
ax.set_yticks([])
plt.imshow(rgb)

In [ ]:
hull7 = ConvexHull(df[df["colony_id"]==7][['cells_i', 'cells_j']].to_numpy())

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.set_xticks([])
ax.set_yticks([])
h = convex_hull_plot_2d(hull7, ax=ax)
plt.imshow(labels_full)

In [ ]:
label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[labels]

In [ ]:
# labels,_,_ = ski.segmentation.relabel_sequential(labels, offset=1)
# labels = da.from_array(labels)
nobjects = np.max(labels)
object_indexes = np.arange(nobjects, dtype=np.int32) + 1
distance = 25
strel = strel_disk(distance)

In [ ]:
def prepare_box_for_contours(box, shape, pad=3):
    """Marginally pads a bounding box so that object boundaries
    are not on cropped image patch edges.
    """
    for i in range(2):
        box[i] = max(0, box[i] - pad)
        box[i+2] = min(shape[i], box[i+2] + pad)
    slices = tuple([slice(box[i], box[i+2]) for i in range(2)])
    top_left = np.array(box[:2])[None] # (1, 2)
    return slices, top_left

def make_polygons_from_mask(mask):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for rp in measure.regionprops(mask):
        # Faster to compute contours on small cell tiles than the whole image
        box_slices, box_top_left = prepare_box_for_contours(list(rp.bbox), mask.shape)
        label_mask = mask[box_slices] == rp.label

        label_cnts = np.concatenate(
            measure.find_contours(label_mask), axis=0
        )

        polygons[rp.label] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_polygons_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        polygons[lab] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_hull_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    hulls = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        hulls[lab] = ConvexHull(label_cnts + box_top_left)
    
    return hulls

def make_delaunay_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    tris = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        tris[lab] = Delaunay(label_cnts + box_top_left)
    
    return tris

def pairwise_polygon_distance(polygons_dict, dist):
    """Computes pairwise distance between all polygons in
    a dictionary. Returns a dictionary of distances.
    """
    polys = list(polygons_dict.values())
    ids = list(polygons_dict.keys())

    tree = STRtree(polys)

    id_map = dict(zip(polys, ids))

    distances = {i: {} for i in ids}

    for poly in polys:
        i = id_map[poly]

        # query nearby polygons using bounding boxes
        candidates = tree.query(poly.buffer(dist))  # 5 px search radius

        for other in candidates:
            j = id_map[other]
            if i == j:
                continue

            d = poly.distance(other)
            distances[i][j] = d
                
    return distances

def get_contour_from_label(mask, labels, df = cell_coords, scaled=True):
    contours = defaultdict(int)
    if not isinstance(labels, list):
        labels = list(labels)
    
    for label in labels:
        box = subimage(mask, df.at[label, "cells_bounds"], pad=5)

        label_cnts = np.concatenate(
        measure.find_contours(box), axis=0
            )
        
        if scaled:
            box_top_left = np.array(df.at[label, "cells_bounds"][:2])[None]
            contours[label] = label_cnts + box_top_left
        else:
            contours[label] = label_cnts
            
    return contours

In [ ]:
objs = ndi.find_objects(labels, max_label=nobjects)
# objs is a list of slice-tuples, one per label 1..nobjects
# missing labels give None

minimums_i = np.empty(nobjects, dtype=np.int64)
maximums_i = np.empty(nobjects, dtype=np.int64)
minimums_j = np.empty(nobjects, dtype=np.int64)
maximums_j = np.empty(nobjects, dtype=np.int64)

for k, slc in enumerate(objs):
    if slc is None:
        minimums_i[k] = 0
        maximums_i[k] = 0
        minimums_j[k] = 0
        maximums_j[k] = 0
        continue

    si, sj = slc
    minimums_i[k] = max(si.start - distance, 0)
    maximums_i[k] = min(si.stop  + distance, labels.shape[0])
    minimums_j[k] = max(sj.start - distance, 0)
    maximums_j[k] = min(sj.stop  + distance, labels.shape[1])

In [ ]:
edges = list()

tPatch = 0
tMask = 0
tExtend = 0
tEdges = 0

def neighbors_for_object(img, object_number, min_i, max_i, min_j, max_j, strel, distance):
    t0 = time.time()

    index = object_number - 1

    patch = img[
        min_i : max_i,
        min_j : max_j,
    ]
    tPatch += (time.time() - t0)
    t0 = time.time()
    #
    # Find the neighbors
    #
    
    patch_mask = patch == (index + 1)
    tMask += (time.time() - t0)
    t0 = time.time()

    if distance <= 5:
        extended = ndi.binary_dilation(patch_mask, strel)
    else:
        extended = (
            scipy.signal.fftconvolve(patch_mask, strel, mode="same") > 0.5
        )
    tExtend += (time.time() - t0)
    t0 = time.time()

    neighbors = np.unique(patch[extended])
    neighbors = neighbors[neighbors != 0]
    neighbors = neighbors[neighbors != object_number]
    if hasattr(neighbors,"compute"):
        neighbors = neighbors.compute()
    edges.extend([(int(n), int(object_number)) for n in neighbors])
    tEdges += (time.time() - t0)

print(f"Make patch: {tPatch} s \nMask patch: {tMask} s\n \
      Dilation/fftconvolve: {tExtend} s\n Add edges: {tEdges} s"
      )

In [ ]:
max_label = dilated_region.max()

label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[dilated_region]

ids = np.unique(colony_img)
ids = ids[ids>0]
cmap = plt.get_cmap("tab20", len(ids))

label_to_color = {
    lab: cmap(i)
    for i, lab in enumerate(ids)
}

# set background / non-colony
label_to_color[-1] = (0.7, 0.7, 0.7, 1)
label_to_color[0] = (0,0,0, 1)

colors_array = np.zeros((max_label + 1, 4))

for lab, color in label_to_color.items():
    colors_array[lab] = color

rgb = colors_array[colony_img]


plt.imshow(rgb)

In [ ]:
colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

cell_coords["colony_id"] = cell_coords.index.map(colony_map)

# sizes = df["colony_id_poly"].value_counts()

# valid = sizes[sizes >= 10].index

# df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
distances = pd.DataFrame(distances_dict)
distances_flat = distances.to_numpy().flatten()
distances_flat = distances_flat[~np.isnan(distances_flat)]

In [ ]:
G = nx.Graph()

# add all cells
G.add_nodes_from(polygons_dict.keys())

threshold = 20  # adjust

for i, neighbors in distances_dict.items():
    for j, d in neighbors.items():
        if d <= threshold:
            G.add_edge(i, j)

components = list(nx.connected_components(G))

colony_map = {}
for k, comp in enumerate(components):
    for cell_id in comp:
        colony_map[cell_id] = k

df["colony_id_poly"] = df.index.map(colony_map)

sizes = df["colony_id_poly"].value_counts()

valid = sizes[sizes >= 10].index

df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
# sizes = df["colony_id"].value_counts()

# valid = sizes[sizes >= 10].index

# df["in_colony"] = df["colony_id"].isin(valid)

# labels = df["colony_id"].values
# unique_labels = np.unique(labels)

# # assign a color to each label
# colors_map = {
#     lab: plt.cm.Spectral(i / len(unique_labels))
#     for i, lab in enumerate(unique_labels)
# }

# # override noise (-1) to black
# filtered_dict = {key:(value if np.isin(key,valid)
#           else (0, 0, 0, 1)) for key, value in colors_map.items()}

# # build color list per point
# colors = [filtered_dict[lab] for lab in labels]

fig, ax = plt.subplots()

ax.scatter(
    df["nucleus_j"],
    df["nucleus_j"],
    c=df["foci_count"],
    s=5
)

ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()

# ax[1].imshow(ski.exposure.rescale_intensity(img[1], in_range=(0,3000)))
# ax[1].set_xticks([])
# ax[1].set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
np.log

In [ ]:
df["log_foci_count"] = np.log1p(df["foci_count"])
points = df[df["colony_id"]==7][['cells_i', 'cells_j']].to_numpy()
hull7 = ConvexHull(points)

In [ ]:
for colony_id, subdf in df.groupby("colony_id"):
    if colony_id == -1:
        continue
    points = subdf[['cells_i', 'cells_j']].to_numpy()
    hull = ConvexHull(points)

    # draw edges
    for simplex in hull.simplices:
        ax.plot(points[simplex, 0], points[simplex, 1],
                color='red', linewidth=1)